# Proyecto 46. Informe exploratorio reproducible con Seaborn

**Bloque:** Seaborn (UT3) · **Tiempo orientativo:** 8–10 horas

## Contexto

Dispones de observaciones **simuladas** de dos canales (`web`, `movil`) y dos versiones (`A`, `B`) de un servicio. Debes elaborar un informe exploratorio reproducible que ayude a describir el equilibrio entre demanda, latencia, errores, satisfacción y energía. No entrenarás modelos ni harás afirmaciones causales.


## Entregables y criterios orientativos

Tu cuaderno debe contener datos cargados desde la ruta local, comprobaciones de calidad, transformaciones justificadas, una tabla de resumen, al menos cinco visualizaciones razonadas y conclusiones limitadas.

Se revisarán: trazabilidad y ejecución completa; elección y legibilidad de figuras; uso correcto de unidades, ejes, leyendas y color; interpretación prudente; y organización clara del cuaderno.


In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import numpy as np

sns.set_theme(style="whitegrid", context="notebook")


def ruta_datos(nombre):
    for ruta in [Path("datos") / "seaborn" / nombre, Path("..") / "datos" / "seaborn" / nombre]:
        if ruta.exists():
            return ruta
    raise FileNotFoundError(f"No se encontró {nombre}. Abre el cuaderno desde la carpeta del curso.")

servicios = pd.read_csv(ruta_datos("46_evaluacion_servicios.csv"))
servicios.head()


,id,canal,version,peticiones_hora,latencia_ms,errores_pct,satisfaccion,energia_wh
0,1,web,A,120,182,1.8,74,15.2
1,2,web,A,135,191,2.0,72,15.9
2,3,web,A,150,205,2.4,70,16.8
3,4,web,A,165,218,2.7,68,17.6
4,5,web,A,180,230,3.0,66,18.5


## Hito 1. Procedencia, esquema y calidad

Lee `datos/seaborn/46_evaluacion_servicios.md`. Muestra el esquema, comprueba filas, tipos, categorías y ausencias. Explica qué representa una fila y qué no puedes inferir de estos datos simulados.


# Cuadernos 39-40

In [ ]:
# Completa el hito 1. 

# 1. Leer la ficha descriptiva y auditar esquema, tipos y calidad
ruta_ficha = ruta_datos("46_evaluacion_servicios.md")
print(ruta_ficha.read_text(encoding="utf-8"))

servicios.info()
print("\nValores ausentes por columna:\n", servicios.isna().sum())
print("\nCategorías en 'canal':", servicios["canal"].unique().tolist())
print("Categorías en 'version':", servicios["version"].unique().tolist())

# 2. Aserciones de contrato de datos
assert servicios.shape == (24, 8), "Se esperaban exactamente 24 filas y 8 columnas"
assert not servicios.isna().any().any(), "Existen valores ausentes"
assert servicios["id"].is_unique, "Los identificadores no son únicos"
assert set(servicios["canal"]) == {"web", "movil"}, "Categorías inesperadas en canal"
assert set(servicios["version"]) == {"A", "B"}, "Categorías inesperadas en version"

| Explicación | Referencia que mejor corresponde | Matiz |
|---|---|---|
| 1. `ruta_ficha.read_text(encoding="utf-8")` lee la ficha local para documentar la procedencia y el carácter sintético de los datos. | **[U09.2]** — lectura de texto con `Path.read_text`; **[U39.2]** — datos locales y rutas portables | La ficha del proyecto es la fuente específica para afirmar que los datos son simulados. |
| 2. `servicios.info()` muestra la estructura de la tabla, los tipos inferidos y los valores no nulos. | **[U39.2]** — primera inspección del conjunto de datos | `info()` es una herramienta de inspección; los tipos concretos dependen del CSV cargado. |
| 3. `servicios.isna().sum()` cuenta los valores ausentes por columna. | **[U40.1]** — inspección de valores ausentes | La unidad recalca que un valor ausente significa desconocido, no un cero medido. |
| 4. `.unique().tolist()` obtiene y muestra los valores distintos de `canal` y `version`. | **[U39.2]** — inspección inicial de los datos | La inspección de categorías es pertinente; la conversión concreta mediante `.tolist()` no se desarrolla como concepto independiente. |
| 5. `assert servicios.shape == (24, 8)` comprueba el número esperado de filas y columnas y detiene la ejecución si no coincide. | **[U18.6]** — forma (`shape`) de un `DataFrame`; **[U39.2, ejercicio 39.01]** — comprobaciones del contrato de datos | El ejercicio 39.01 comprueba condiciones de datos con aserciones; el concepto de `shape` se explica en U18.6. |
| 6. `assert not servicios.isna().any().any()` falla si existe algún valor ausente en la tabla. | **[U39.2, ejercicio 39.01]** — aserción equivalente sobre valores ausentes; **[U40.1]** — significado y revisión de `NaN` | La expresión `.any().any()` es una aplicación para reducir las comprobaciones a un único booleano. |
| 7. `assert servicios["id"].is_unique` comprueba que cada identificador aparece una sola vez. | **[U19.5]** — comprobación de unicidad de identificadores | La unidad incluye una aserción equivalente para validar identificadores únicos. |
| 8. `assert set(servicios["canal"]) == {"web", "movil"}` comprueba que aparecen exactamente las categorías previstas. | **[U08.4]** — conjuntos y elementos únicos; **[U39.2, ejercicio 39.01]** — validación del esquema | Aplicación de conjuntos para contrastar las categorías observadas con las esperadas. |
| 9. `assert set(servicios["version"]) == {"A", "B"}` valida las categorías de versión; cada fila corresponde a una observación simulada, no a una medición de un servicio real. | **[U08.4]** — conjuntos; **[U39.2]** — inspección de datos y límites de interpretación | La condición de datos simulados y el significado de cada fila se verifican en la ficha y el contexto del Proyecto 46. |

## Hito 2. Transformación documentada

Crea un `DataFrame` de análisis sin modificar `servicios`. Añade al menos una columna derivada vectorizada y útil —por ejemplo, peticiones por Wh o latencia por petición—. Justifica por qué tiene sentido y evita divisiones no comprobadas.


# Cuaderno 45

In [ ]:
# Completa el hito 2. 

# Crear copia independiente para no alterar el DataFrame original 'servicios'
analisis = servicios.copy()

# Verificar denominadores > 0 antes de aplicar operaciones vectorizadas
assert (analisis["energia_wh"] > 0).all(), "Existen registros con energia_wh <= 0"
assert (analisis["peticiones_hora"] > 0).all(), "Existen registros con peticiones_hora <= 0"

analisis = analisis.assign(
    peticiones_por_wh=analisis["peticiones_hora"] / analisis["energia_wh"],
    latencia_por_peticion_ms=analisis["latencia_ms"] / analisis["peticiones_hora"],
)

# Confirmar que la tabla original permanece intacta
assert servicios.shape == (24, 8)
display(analisis[["id", "canal", "version", "peticiones_por_wh", "latencia_por_peticion_ms"]].head())

| Explicación | Referencia que mejor corresponde | Matiz |
|---|---|---|
| 1. `analisis = servicios.copy()` crea una copia para trabajar sin modificar la tabla original. | **[U45.1]** — flujo de transformación trazable sin alterar los datos crudos | Corresponde directamente a la recomendación de conservar los datos originales. |
| 2. `assert (analisis["energia_wh"] > 0).all()` comprueba que todos los valores de energía sean positivos antes de usarlos en una división. | **[U45.1]** — validar los denominadores antes de dividir | La unidad usa este patrón para comprobar denominadores. |
| 3. `assert (analisis["peticiones_hora"] > 0).all()` comprueba que la demanda sea positiva antes de usarla como divisor. | **[U45.1]** — validar los denominadores antes de dividir | La aserción evita divisiones por cero o por valores negativos. |
| 4. `analisis.assign(...)` añade columnas calculadas mediante operaciones vectorizadas y devuelve un `DataFrame` transformado. | **[U45.1]** — transformaciones vectorizadas con Pandas | `assign` permite hacer explícitas las transformaciones y mantener la tabla original. |
| 5. `peticiones_hora / energia_wh` calcula el cociente entre la tasa de peticiones por hora y el consumo energético. | **[U45.1]** — creación de columnas derivadas vectorizadas | No es estrictamente “peticiones por Wh”: sus unidades son peticiones/(hora·Wh). No conviene llamarlo eficiencia ni interpretarlo como peticiones por Wh sin redefinir la medida o el intervalo. |
| 6. `latencia_ms / peticiones_hora` calcula el cociente entre la latencia y la tasa horaria de peticiones. | **[U45.1]** — operaciones vectorizadas | No equivale a latencia por petición: sus unidades serían ms·hora/petición. Para obtener latencia por petición se necesitaría una medida compatible de peticiones en el mismo intervalo. |
| 7. `assert servicios.shape == (24, 8)` verifica que la tabla original mantiene sus dimensiones. | **[U18.6]** — forma de un `DataFrame`; **[U45.1]** — preservar los datos originales | La aserción verifica la forma, aunque por sí sola no demuestra que todos los valores originales permanezcan intactos. |
| 8. `display(analisis[[...]].head())` selecciona columnas pertinentes y muestra las primeras filas para inspeccionar el resultado. | **[U39.2]** — inspección inicial de tablas; **[U45.1]** — revisión de una transformación | `head()` presenta una muestra, no valida por sí sola todas las filas. |

## Hito 3. Resumen y distribución

Construye una tabla de medias por canal y versión. Después crea una figura univariante o categórica que conserve observaciones individuales y una figura de distribución. Explica qué resume cada una.


# Cuadernos 40-42

In [ ]:
# Completa el hito 3.

# 1. Tabla de medias por canal y versión
tabla_resumen = (
    analisis.groupby(["canal", "version"], as_index=False)[
        ["peticiones_hora", "latencia_ms", "errores_pct", "satisfaccion", "energia_wh", "peticiones_por_wh"]
    ]
    .mean()
    .round(2)
)
display(tabla_resumen)

# 2. Panel con gráfico categórico + puntos individuales (izq.) y distribución acumulada ECDF (der.)
fig, ejes = plt.subplots(1, 2, figsize=(12.5, 4.6), layout="constrained")

# Visualización 1: Boxplot + Stripplot
sns.boxplot(
    data=analisis, x="canal", y="latencia_ms", hue="version",
    palette="pastel", width=0.5, fliersize=0, ax=ejes[0]
)
sns.stripplot(
    data=analisis, x="canal", y="latencia_ms", hue="version",
    dodge=True, color="#222222", alpha=0.8, marker="o", ax=ejes[0]
)
handles, labels = ejes[0].get_legend_handles_labels()
ejes[0].legend(handles[:2], labels[:2], title="Versión")
ejes[0].set(
    title="Latencia por canal y versión (resumen y puntos)",
    xlabel="Canal",
    ylabel="Latencia (ms)"
)

# Visualización 2: ECDF por versión
sns.ecdfplot(
    data=analisis, x="satisfaccion", hue="version",
    linewidth=2.2, ax=ejes[1]
)
ejes[1].set(
    title="Distribución acumulada (ECDF) de satisfacción",
    xlabel="Satisfacción (puntos)",
    ylabel="Proporción acumulada"
)
ejes[1].legend(title="Versión")

fig.suptitle("Resumen categórico de latencia y distribución de satisfacción")
plt.show()

| Explicación | Referencia que mejor corresponde | Matiz |
|---|---|---|
| 1. `groupby(["canal", "version"])`, `.mean()` y `.round(2)` calculan y redondean las medias de las variables seleccionadas por combinación de grupos. | **[U45.2]** — resumir datos con `groupby` y agregaciones | La tabla resume grupos y no sustituye a las observaciones individuales. |
| 2. `display(tabla_resumen)` muestra la tabla agregada antes de representar los datos. | **[U45.2]** — revisión de resúmenes; **[U39.2]** — visualización de tablas | Presentar la tabla ayuda a revisar los valores resumidos. |
| 3. `plt.subplots(1, 2, ..., layout="constrained")` crea una figura de Matplotlib con dos ejes para organizar las visualizaciones. | **[U42.3]** — composición de visualizaciones categóricas; **[U39.4]** — funciones Seaborn de nivel de ejes | U39.4 explica cómo integrar gráficos Seaborn en ejes de Matplotlib; la figura con dos paneles es una aplicación de esa composición. |
| 4. `sns.boxplot(...)` resume la distribución de latencia por canal y versión mediante cajas. | **[U42.3]** — distribución de datos dentro de grupos | El boxplot representa mediana, cuartiles y valores alejados según la regla del gráfico. |
| 5. `sns.stripplot(..., dodge=True, ...)` superpone puntos individuales desplazados por grupo. | **[U42.3]** — `stripplot` y observaciones individuales | Los puntos permiten ver las observaciones que el resumen del boxplot no muestra por separado. |
| 6. `get_legend_handles_labels()` y `legend(handles[:2], labels[:2], ...)` conservan las entradas necesarias y evitan duplicar la leyenda al superponer dos capas. | **[U42.3]** — leyenda al combinar boxplot y stripplot | La unidad incluye este patrón para eliminar las entradas duplicadas de la leyenda. |
| 7. `ejes[0].set(...)` asigna el título y las etiquetas del primer panel, incluida la unidad de latencia. | **[U39.4]** — configuración de ejes para gráficos Seaborn de nivel de ejes | La unidad presenta el uso de ejes Matplotlib para configurar figuras que contienen gráficos Seaborn. |
| 8. `sns.ecdfplot(..., hue="version")` muestra la distribución acumulada de satisfacción, separada por versión. | **[U40.4]** — función de distribución acumulada empírica (ECDF) | La ECDF muestra proporciones acumuladas sin depender de intervalos de histograma ni de suavizado KDE. |
| 9. `ejes[1].set(...)` y `ejes[1].legend(...)` configuran el título, las etiquetas y la leyenda del segundo panel. | **[U40.4]** — interpretación de ECDF; **[U39.4]** — configuración de ejes | La unidad 40 cubre la lectura de la ECDF; la configuración del eje sigue el patrón de nivel de ejes. |
| 10. `fig.suptitle(...)` añade el título global y `plt.show()` muestra la figura. | **[U44.1]** — títulos globales de figuras facetadas; **[U42.3]** — figuras con varios paneles | La figura de este proyecto se crea con Matplotlib; el título global es el mismo recurso de figura usado en los ejemplos de Seaborn. |

## Hito 4. Relaciones y matrices

Crea una dispersión que relacione demanda y latencia; distingue versión con color y estilo. Añade una matriz de correlaciones o una tabla pivote con `heatmap`. Indica qué asociación observas y por qué no implica causalidad.


# Cuadernos 39-41-43

In [ ]:
# Completa el hito 4.

fig, ejes = plt.subplots(1, 2, figsize=(13, 4.8), layout="constrained")

# Visualización 3: Dispersión demanda vs. latencia diferenciando versión por color y estilo
sns.scatterplot(
    data=analisis,
    x="peticiones_hora",
    y="latencia_ms",
    hue="version",
    style="version",
    markers={"A": "o", "B": "s"},
    s=85,
    ax=ejes[0],
)
ejes[0].set(
    title="Demanda frente a latencia por versión",
    xlabel="Demanda (peticiones/hora)",
    ylabel="Latencia (ms)",
)
ejes[0].legend(title="Versión")

# Visualización 4: Mapa de calor de correlaciones lineales con máscara triangular
cols_num = ["peticiones_hora", "latencia_ms", "errores_pct", "satisfaccion", "energia_wh", "peticiones_por_wh"]
corr = analisis[cols_num].corr()
mascara = np.triu(np.ones_like(corr, dtype=bool), k=1)

sns.heatmap(
    corr,
    mask=mascara,
    annot=True,
    fmt=".2f",
    cmap="vlag",
    vmin=-1,
    vmax=1,
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={"label": "Correlación de Pearson"},
    ax=ejes[1],
)
ejes[1].set_title("Matriz de correlaciones (triángulo inferior)")

fig.suptitle("Análisis de relaciones bivariantes y matriz de correlación")
plt.show()

| Explicación | Referencia que mejor corresponde | Matiz |
|---|---|---|
| 1. `plt.subplots(1, 2, ..., layout="constrained")` crea una figura con dos paneles para la dispersión y el mapa de calor. | **[U42.3]** — composición de varios gráficos; **[U39.4]** — gráficos Seaborn de nivel de ejes | U39.4 respalda el uso de ejes de Matplotlib con Seaborn; los dos paneles aplican ese patrón. |
| 2. `sns.scatterplot(...)` representa cada observación de demanda y latencia, diferenciando la versión mediante color y marcador. | **[U41.1]** — diagramas de dispersión y mapeos `hue`/`style`; **[U39.3]** — gramática visual | Usar color y forma para la misma categoría evita depender solo del color. |
| 3. `ejes[0].set(...)` y `legend(...)` añaden título, unidades y leyenda a la dispersión. | **[U41.1]** — presentación de relaciones entre variables | Las etiquetas identifican las variables y unidades; la leyenda identifica la versión. |
| 4. `cols_num = [...]` selecciona las columnas numéricas que se incluirán en el análisis de correlación. | **[U43.1]** — selección de variables numéricas para correlaciones | Excluir identificadores y categorías evita tratarlos como medidas numéricas. |
| 5. `analisis[cols_num].corr()` calcula la matriz de correlaciones lineales entre las columnas seleccionadas. | **[U43.1]** — correlación y matriz de correlaciones | La correlación describe asociación lineal en los datos; no demuestra causalidad. |
| 6. `np.triu(np.ones_like(corr, dtype=bool), k=1)` crea una máscara para ocultar el triángulo superior redundante. | **[U43.2]** — máscaras triangulares para matrices simétricas | La matriz de correlación es simétrica, por lo que se puede ocultar un triángulo para facilitar la lectura. |
| 7. `sns.heatmap(...)` representa las correlaciones con anotaciones, paleta divergente, límites de −1 a 1 y centro en cero. | **[U43.2]** — escala, anotaciones y máscara de un `heatmap` | Es una escala apropiada para correlaciones que pueden ser positivas o negativas. |
| 8. `set_title(...)`, `fig.suptitle(...)` y `plt.show()` titulan y muestran la figura; las correlaciones se interpretan como asociaciones observadas, no como causas. | **[U43.1]** — límites de interpretación de la correlación; **[U44.1]** — título global | No incluyas coeficientes concretos en la explicación salvo que se hayan comprobado en la matriz calculada. Además, la interpretación de las columnas derivadas depende de que sus fórmulas y unidades sean válidas. |

## Hito 5. Facetado y comunicación

Construye un gráfico facetado por canal usando `relplot`, `catplot` o `FacetGrid`. La figura debe mantener escalas comparables, etiquetas completas y una leyenda legible. Exporta una de tus figuras a `proyectos/salidas/46_informe_seaborn.png` mediante una ruta relativa, pero no añadas el PNG al repositorio.


In [ ]:
# Completa el hito 5.

# Comprobación previa de que no hay valores repetidos por (canal, version, peticiones_hora)
assert analisis.groupby(["canal", "version", "peticiones_hora"]).size().eq(1).all()

# Visualización 5: Facetado por canal con función de nivel de figura relplot
grafico_facetado = sns.relplot(
    data=analisis,
    x="peticiones_hora",
    y="energia_wh",
    hue="version",
    style="version",
    col="canal",
    kind="line",
    markers={"A": "o", "B": "s"},
    dashes={"A": "", "B": (4, 2)},
    height=4.2,
    aspect=1.15,
)

grafico_facetado.set_axis_labels("Demanda (peticiones/hora)", "Consumo energético (Wh)")
grafico_facetado.set_titles("Canal: {col_name}")
grafico_facetado.legend.set_title("Versión")
grafico_facetado.figure.suptitle("Consumo energético según demanda, canal y versión", y=1.04)

# Exportación reproducible mediante ruta relativa
ruta_salida = Path("proyectos") / "salidas" / "46_informe_seaborn.png"
ruta_salida.parent.mkdir(parents=True, exist_ok=True)
grafico_facetado.savefig(ruta_salida, dpi=150, bbox_inches="tight")

assert ruta_salida.exists(), f"No se generó el archivo en {ruta_salida}"
print("Figura exportada correctamente en:", ruta_salida)
plt.show()

| Explicación | Referencia que mejor corresponde | Matiz |
|---|---|---|
| 1. `groupby(...).size().eq(1).all()` comprueba que hay una observación por combinación de canal, versión y demanda antes de dibujar líneas. | **[U41.2]** — repeticiones y agregación implícita de `lineplot` | La unidad explica que `lineplot` puede agregar registros repetidos y muestra una comprobación equivalente. |
| 2. `sns.relplot(..., col="canal", kind="line")` crea un panel por canal y representa la relación entre demanda y energía. | **[U44.1]** — facetado con `relplot` | `relplot` es una función de nivel de figura que permite crear paneles mediante `col`. |
| 3. `hue`, `style`, `markers` y `dashes` distinguen las versiones por color, marcador y tipo de línea. | **[U39.3]** — gramática visual y mapeos; **[U44.1]** — mapeos en gráficos facetados | La doble codificación facilita distinguir grupos sin depender únicamente del color. |
| 4. `set_axis_labels(...)` establece las etiquetas compartidas de los ejes con sus unidades. | **[U44.1]** — etiquetas en gráficos `relplot` facetados | Es el método propio del objeto de nivel de figura generado por Seaborn. |
| 5. `set_titles(...)` asigna a cada panel un título que incluye el canal. | **[U44.1]** — títulos de paneles facetados | `{col_name}` se sustituye por el nombre del canal de cada panel. |
| 6. `legend.set_title("Versión")` cambia el título de la leyenda del gráfico facetado. | **[U44.1–U44.2]** — configuración de leyendas en `relplot` y `FacetGrid` | La leyenda pertenece al objeto de figura/grilla de Seaborn. |
| 7. `grafico_facetado.figure.suptitle(...)` añade un título global a la figura. | **[U44.1]** — título global de una figura facetada | El objeto `relplot` permite acceder a la figura subyacente mediante `.figure`. |
| 8. `Path("proyectos") / "salidas" / ...` construye una ruta relativa y portable para la figura. | **[U45.4]** — exportar sin rutas absolutas | `Path` evita depender de una ruta personal del equipo. |
| 9. `ruta_salida.parent.mkdir(parents=True, exist_ok=True)` crea la carpeta de destino y las intermedias si faltan. | **[U45.4]** — preparar una ruta relativa de exportación | `parents=True` amplía el patrón al crear también los directorios superiores necesarios. |
| 10. `savefig(...)` exporta la figura; `exists()` verifica que se generó, y `print(...)`/`plt.show()` informan de la ruta y muestran el gráfico. | **[U45.4]** — exportación; **[U35.4]** — exportación comprobada | U45.4 cubre las rutas y opciones de guardado; el ejercicio 35.04 muestra una comprobación explícita de que el archivo existe. |

## Hito 6. Conclusiones

Redacta entre cinco y ocho frases que incluyan: dos pautas observadas, una comparación entre versiones o canales, dos límites de interpretación y una comprobación adicional necesaria antes de tomar una decisión sobre un servicio real.

## Lista final

- [ ] El cuaderno se ejecuta desde cero sin conexión a Internet.
- [ ] Todas las figuras tienen título, etiquetas, unidades y leyenda cuando corresponde.
- [ ] El color no es la única señal para distinguir grupos.
- [ ] Las transformaciones y agregaciones son explícitas.
- [ ] Las conclusiones separan datos observados de hipótesis.


### Conclusiones

En las 24 observaciones analizadas se identifica una primera pauta clara: a medida que la demanda aumenta de 120 a 195 peticiones por hora en todos los grupos, crecen de forma paralela la latencia, el porcentaje de errores y el consumo energético. Como segunda pauta, la latencia y los errores mantienen una asociación negativa muy estrecha con la satisfacción ($r = -0{,}99$ y $r = -0{,}98$, respectivamente). Al comparar canales y versiones, el canal `web` con la versión `B` obtiene los mejores registros medios en latencia (189,83 ms), errores (2,05 %) y satisfacción (73,67 puntos), aunque presenta el mayor consumo energético medio (18,50 Wh frente a los 15,77 Wh de `movil` versión `A`). No obstante, estas observaciones tienen un primer límite fundamental: todos los datos son sintéticos y simulados para el curso, por lo que no representan el comportamiento de una infraestructura real. Como segundo límite, las asociaciones detectadas reflejan únicamente covariación sobre seis niveles fijos de carga y no demuestran causalidad ni permiten extrapolar fuera del rango de 120 a 195 peticiones por hora. Antes de tomar cualquier decisión técnica en producción, sería necesario ejecutar pruebas de carga controladas en preproducción midiendo telemetría real con variabilidad temporal y uso de recursos (CPU y memoria).

### Resumen de funciones Seaborn (Unidades 39–45) fuera del Proyecto 46
- **[U40.2 / U40.3]** `sns.histplot(..., stat="density")` y `sns.kdeplot(...)` dibujan distribuciones normalizadas por área (`1`) donde el eje vertical mide densidad y no conteos,
- **[U40.5]** `sns.displot(...)` crea una figura completa (nivel de figura) para histogramas, KDE o ECDF divididos en paneles,
- **[U41.3]** `sns.jointplot(...)` combina un gráfico bivariante central con distribuciones marginales en los bordes,
- **[U41.4]** `sns.regplot(...)` (nivel de ejes) y `sns.lmplot(...)` (nivel de figura con `hue`/`col`) ajustan una recta lineal descriptiva con banda de incertidumbre,
- **[U42.1 / U42.2]** `sns.countplot(...)` cuenta filas por categoría (nunca medias) y `sns.barplot(..., errorbar=("ci", 90))` estima medias con intervalo de incertidumbre,
- **[U42.3 / U42.4]** `sns.violinplot(...)`, `sns.swarmplot(...)` y `sns.catplot(...)` representan distribuciones categóricas completas y facetadas,
- **[U43.4]** `sns.clustermap(...)` reordena filas y columnas por similitud jerárquica exploratoria creando su propia figura,
- **[U44.3]** `sns.pairplot(..., vars=[...], corner=True)` genera una matriz de relaciones entre pocas variables seleccionadas ocultando el triángulo duplicado.

## Trampas

### 1. Nivel de ejes vs. Nivel de figura
- **[U39.4 · U40.5 · U41.3 · U42.4]** **Nivel de ejes** (`lineplot`, `scatterplot`, `histplot`, `kdeplot`, `ecdfplot`, `regplot`, `countplot`, `barplot`, `boxplot`, `violinplot`, `stripplot`, `swarmplot`, `heatmap`): aceptan `ax=ax`, dibujan dentro de una figura de Matplotlib ya creada y se configuran con `ax.set(...)`.
- **[U39.4 · U40.5 · U41.3 · U42.4 · U44.1]** **Nivel de figura** (`relplot`, `displot`, `jointplot`, `lmplot`, `catplot`, `clustermap`, `pairplot`): crean su propia figura o cuadrícula (`FacetGrid` / `PairGrid`), **no aceptan `ax=`** y se configuran con `g.set_axis_labels(...)` y `g.figure.suptitle(...)`.

---

### 2. Qué mide cada gráfico de distribución y categórico
- **[U40.2 · U40.3]** **`histplot(..., stat="count")`**: mide **conteo de observaciones**; con `stat="density"` (y en `kdeplot`) mide **densidad** (el área total suma `1`, no la altura).
- **[U40.4]** **`ecdfplot`**: muestra la **proporción acumulada** (`0` a `1`) sin depender de elegir `bins` ni suavizado KDE.
- **[U42.1 · U42.2]** **`countplot` vs. `barplot`**: `countplot` cuenta **filas por categoría** (nunca calcula medias); `barplot` estima la **media** con un intervalo de confianza (pero oculta la dispersión individual).

---

### 3. Correlación, ajustes y causalidad
- **[U39.1 · U41.1 · U43.1]** **Asociación $\neq$ Causalidad**: una correlación alta ($r = 0{,}80$ o $r = -0{,}99$), una diferencia entre grupos o una recta de `regplot`/`lmplot` muestran asociación o tendencia descriptiva en la muestra observada, **nunca causalidad** ni predicción validada.
- **[U41.4]** **Banda sombreada de `regplot` / `lmplot`**: representa la **incertidumbre de la estimación de la recta**, no un intervalo que contenga todos los datos.

---

### 4. Matrices, tablas pivote y accesibilidad
- **[U43.1 · U43.2]** **Matriz de correlación**: la diagonal vale `1` y `np.triu(..., k=1)` oculta el triángulo superior duplicado; se representa con paleta divergente centrada en cero (`vmin=-1, vmax=1, center=0`).
- **[U43.3]** **Matriz de confusión (`pd.crosstab`)**: conviene fijar explícitamente qué eje son las **clases reales** (filas) y cuál las **predichas** (columnas).
- **[U39.3 · U41.1 · U42.3 · U44.1]** **Accesibilidad visual**: el color (`hue`) nunca debe ser la única señal para distinguir grupos; acompáñalo de `style` (marcadores/trazos), posición (`dodge=True`) o paneles (`col=`).

---

### 5. Calidad de datos y reproducibilidad
- **[U40.1]** **Ausentes (`NaN`)**: significan dato desconocido, **nunca equivalen a un `0` medido**.
- **[U45.1 · U45.4]** **Trazabilidad y exportación**: nunca modifiques el `DataFrame` crudo directamente (usa `.copy()` o `.assign()`), comprueba denominadores $> 0$ antes de dividir y exporta siempre con rutas relativas (`Path(...)`) y `bbox_inches="tight"`.